# Automix training on Kaggle

Requirements before running (see `KAGGLE.md` in the repo):
- **Accelerator: GPU** (Settings panel on the right -> Accelerator -> GPU T4/P100)
- The raw-data dataset attached as input (default name assumed: `automix-raw`)

Safe to re-run top to bottom: training auto-resumes from the last checkpoint if one exists.

In [ ]:
# 1) Code + dependencies (torch/torchaudio are pinned; ~5 min install)
# NOTE: clone the working branch explicitly - a bare clone gets the default
# branch (main), which does not have the current training code.
BRANCH = 'mac-support_testrun'
%cd /kaggle/working
!rm -rf automatic_mixing
!git clone -b {BRANCH} https://github.com/takyol/automatic_mixing.git
%cd automatic_mixing
!git log --oneline -1
!pip install -q -e .
import torch; print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 2) Prepare data into /kaggle/working/data_processed (skips if already done this session)
import os, yaml

RAW = '/kaggle/input/automix-raw'  # adjust if your dataset has a different name
assert os.path.isdir(RAW), f'{RAW} not found - attach the dataset (Add Input) or fix RAW'

if not os.path.isdir('/kaggle/working/data_processed'):
    cfg = yaml.safe_load(open('configs/spheres_prep.yaml'))
    cfg['raw_root'] = f'{RAW}/spheres'
    cfg['output_root'] = '/kaggle/working/data_processed'
    yaml.dump(cfg, open('/kaggle/working/spheres_prep.yaml', 'w'))

    cfg = yaml.safe_load(open('configs/synthsod_prep.yaml'))
    cfg['raw_root'] = f'{RAW}/synthsod/SynthSOD/SynthSOD_data'
    cfg['output_root'] = '/kaggle/working/data_processed'
    yaml.dump(cfg, open('/kaggle/working/synthsod_prep.yaml', 'w'))

    !python scripts/prepare_spheres.py --config /kaggle/working/spheres_prep.yaml
    !python scripts/prepare_synthsod.py --config /kaggle/working/synthsod_prep.yaml

print(len(os.listdir('/kaggle/working/data_processed')), 'songs prepared')

In [ ]:
# 3) Train - auto-resumes from the most recent run's checkpoint if one exists.
# Each fresh run gets its own <date>_<time>_kaggle folder; a resumed run
# continues in that same folder (train.py derives the run name from --resume).
import glob, os
cmd = 'python -u scripts/train.py --config configs/kaggle.yaml'
ckpts = glob.glob('/kaggle/working/checkpoints/*/last.pt')
if ckpts:
    resume = max(ckpts, key=os.path.getmtime)
    cmd += f' --resume {resume}'
    print('resuming from', resume)
else:
    print('starting a fresh run')
print(cmd)
!{cmd}

After the run: `/kaggle/working/checkpoints/kaggle/` holds `best.pt` / `last.pt`,
`/kaggle/working/runs/kaggle/` the TensorBoard log. Use **Save Version -> Save & Run All**
to train headless (up to 12h); outputs then appear on the notebook's Output tab for download.